In [8]:
import re
from pathlib import Path
import os

import ee

import gee_export_helpers as gee_helpers


#####################################
# PARAMETERS
#####################################

CYVERSE = False

# Can be:
# - a folder/asset path containing images
# - an ImageCollection asset path
DATA_LOCATION = "projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev"

GEE_PROJECT = "ee-tymc5571-cbi-module"
DRIVE_FOLDER = "GEE_Exports_combinedwildland"

CRS = "EPSG:5070"
EXPORT_SCALE = 30

# "images" = export all image assets under DATA_LOCATION
# "imagecollection" = export all images inside an ImageCollection asset
EXPORT_TYPE = "imagecollection"

# Optional filter applied to asset IDs
IMAGE_STRING = None

# String used in export names; can be anything descriptive of the data being exported
EXPORT_STRING = "combinedwildland"

# Property used to build export names
# Falls back to system:index, then asset basename if missing
EXPORT_NAME_PROPERTY = "year"

# Optional limit for testing
MAX_EXPORTS = None

N_WORKERS = 6
PREDICTOR = 3  # use 2 for integer rasters, 3 for float rasters
DO_EXPORT = False
DO_DOWNLOAD = True
EXPORT_REGION = None # ee.Geometry.Polygon(...) or None for full extent
NEW_DATA_DIRECTORY = True


#####################################
# LOCAL PATHS
#####################################

if CYVERSE:
    SERVICE_FILE = "~/data-store/data/iplant/home/tylermcintosh/secrets/tymc5571-utils-project-692f27c034dd.json"
    DOWNLOAD_LOCATION = "~/data-store/data/iplant/home/shared/earthlab/macrosystems/cbi-module/data/derived/"
else:
    SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"
    DOWNLOAD_LOCATION = "C:/Users/tymc5571/dev/cbi-module/data/derived/"

Path(DOWNLOAD_LOCATION).mkdir(parents=True, exist_ok=True)

if NEW_DATA_DIRECTORY:
    # Create a new directory for the merged data
    DOWNLOAD_LOCATION = os.path.join(DOWNLOAD_LOCATION, EXPORT_STRING)
    Path(DOWNLOAD_LOCATION).mkdir(parents=True, exist_ok=True)


#####################################
# EE AUTH / INIT
#####################################

ee.Authenticate(
    auth_mode="notebook",
    scopes=[
        "https://www.googleapis.com/auth/earthengine",
        "https://www.googleapis.com/auth/devstorage.read_write",
        "https://www.googleapis.com/auth/drive",
    ],
)
ee.Initialize(project=GEE_PROJECT)






In [9]:

#####################################
# DISCOVER ASSETS
#####################################

if EXPORT_TYPE == "images":
    assets = gee_helpers.get_assets_from_folder(DATA_LOCATION, IMAGE_STRING)

elif EXPORT_TYPE == "imagecollection":
    assets = gee_helpers.get_assets_from_imagecollection(DATA_LOCATION, IMAGE_STRING)

else:
    raise ValueError("EXPORT_TYPE must be either 'images' or 'imagecollection'")

assets = sorted(assets)

if MAX_EXPORTS is not None:
    assets = assets[:MAX_EXPORTS]

print(f"Found {len(assets)} assets")
for a in assets:
    print(a)



Found 36 assets
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1985
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1986
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1987
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1988
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1989
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1990
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1991
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1992
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1993
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/combinedwildland_allsev_1994
projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev/c

In [27]:
import importlib
importlib.reload(gee_helpers)

<module 'gee_export_helpers' from 'c:\\Users\\tymc5571\\dev\\cbi-module\\code\\gee_export_helpers.py'>

In [5]:

#####################################
# EXPORT TO GDRIVE
#####################################

for asset_id in assets:
    image = ee.Image(asset_id)

    export_property_data = gee_helpers.get_export_name(
        image=image,
        asset_id=asset_id,
        export_name_property=EXPORT_NAME_PROPERTY,
    )

    export_name = gee_helpers.sanitize_name(f"{EXPORT_STRING}_{export_property_data}")

    region = EXPORT_REGION if EXPORT_REGION is not None else image.geometry().bounds()

    if DO_EXPORT:
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=export_name,
            folder=DRIVE_FOLDER,
            fileNamePrefix=export_name,
            region=region,
            scale=EXPORT_SCALE,
            crs=CRS,
            maxPixels=1e13,
        )
        task.start()
        print(f"Started export: {export_name}")

    else: 
        print(f"Would have started: {export_name}")

Would have started: combinedwildland_1985
Would have started: combinedwildland_1986
Would have started: combinedwildland_1987
Would have started: combinedwildland_1988
Would have started: combinedwildland_1989
Would have started: combinedwildland_1990
Would have started: combinedwildland_1991
Would have started: combinedwildland_1992
Would have started: combinedwildland_1993
Would have started: combinedwildland_1994
Would have started: combinedwildland_1995
Would have started: combinedwildland_1996
Would have started: combinedwildland_1997
Would have started: combinedwildland_1998
Would have started: combinedwildland_1999
Would have started: combinedwildland_2000
Would have started: combinedwildland_2001
Would have started: combinedwildland_2002
Would have started: combinedwildland_2003
Would have started: combinedwildland_2004
Would have started: combinedwildland_2005
Would have started: combinedwildland_2006
Would have started: combinedwildland_2007
Would have started: combinedwildla

In [11]:
#####################################
# READ AND MERGE TO LOCAL
#####################################

for asset_id in assets:
    image = ee.Image(asset_id)

    export_property_data = gee_helpers.get_export_name(
        image=image,
        asset_id=asset_id,
        export_name_property=EXPORT_NAME_PROPERTY,
    )

    export_name = gee_helpers.sanitize_name(f"{EXPORT_STRING}_{export_property_data}")


    if DO_DOWNLOAD:

        gee_helpers.download_merge_from_drive(
            description=export_name,
            local_filename=os.path.join(DOWNLOAD_LOCATION, f'{export_name}.tif'),
            drive_folder=DRIVE_FOLDER,
            service_account_file=SERVICE_FILE,
            n_workers = N_WORKERS,
            as_cog=True,
            compress="ZSTD",
            predictor=PREDICTOR,
            blocksize=512,
            bigtiff="IF_SAFER",
            check_existing=True,
            #dst_nodata=-9999.0,
            dataset_metadata={
                "TITLE": f"WestWiSe {EXPORT_STRING} {export_property_data}",
                "SUMMARY": f"Annual western-US wildfire severity raster for {export_property_data}",
                "CREATOR": "Tyler L. McIntosh",
                "INSTITUTION": "University of Colorado Boulder",
                "LICENSE": "CC-BY-NC-4.0",
                "CONTACT": "tyler.l.mcintosh@colorado.edu",
                "PROCESSING_LINEAGE": "Computed in GEE using Parks et al 2019, exported to Google Drive, downloaded from Google Drive and mosaicked with GDAL",
            },
        )

    else:
        print(f"Would have downloaded: {export_name}. Title: WestWiSe {EXPORT_STRING} {export_property_data}.")

Found 2 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmprawrkevh ...
Downloaded combinedwildland_1985-0000000000-0000082944.tif
Downloaded combinedwildland_1985-0000018432-0000027648.tif
Confirmed band descriptions are consistent across downloaded tile(s).
Building VRT from 2 tile(s) ...
Writing COG via gdal.Translate...
Final output saved to: C:\Users\tymc5571\dev\cbi-module\data\derived\combinedwildland\combinedwildland_1985.tif
Cleaned up temporary directory: C:\Users\tymc5571\AppData\Local\Temp\tmprawrkevh
Found 110 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmpoz8isi3v ...
Downloaded combinedwildland_1986-0000000000-0000018432.tif
Downloaded combinedwildland_1986-0000000000-0000046080.tif
Downloaded combinedwildland_1986-0000000000-0000036864.tif
Downloaded combinedwildland_1986-0000000000-0000027648.tif
Downloaded combinedwildland_1986-0000000000-0000009216.tif
Downloaded combinedwildland_1986-0000000000-0000000000.tif
Downloaded comb

In [ ]:





# for asset_id in assets:
#     image = ee.Image(asset_id)

#     export_property_data = gee_helpers.get_export_name(
#         image=image,
#         asset_id=asset_id,
#         export_name_property=EXPORT_NAME_PROPERTY,
#     )

#     export_name = gee_helpers.sanitize_name(f"{EXPORT_STRING}_{export_property_data}")


#     if DO_DOWNLOAD:
#         gee_helpers.download_merge_from_drive(
#             description=export_name,
#             local_filename=os.path.join(DOWNLOAD_LOCATION, f'{export_name}.tif'),
#             drive_folder=DRIVE_FOLDER,
#             service_account_file=SERVICE_FILE,
#             as_cog=False,
#             compress="ZSTD",
#             predictor=PREDICTOR,
#             blocksize=512, # tile size; 256–512 is good default
#             bigtiff="IF_SAFER",
#             n_workers=N_WORKERS,
#             check_existing=True,
#             verbose=True
#         )

#     else:
#         print(f"Would have downloaded: {export_name}")

Found 144 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmpvgan_au4 ...
Downloaded WUMI2024a_1993-0000000000-0000036864.tif
Downloaded WUMI2024a_1993-0000000000-0000046080.tif
Downloaded WUMI2024a_1993-0000000000-0000009216.tif
Downloaded WUMI2024a_1993-0000000000-0000027648.tif
Downloaded WUMI2024a_1993-0000000000-0000055296.tifDownloaded WUMI2024a_1993-0000000000-0000000000.tif

Downloaded WUMI2024a_1993-0000000000-0000018432.tif
Downloaded WUMI2024a_1993-0000000000-0000064512.tif
Downloaded WUMI2024a_1993-0000000000-0000073728.tif
Downloaded WUMI2024a_1993-0000000000-0000101376.tif
Downloaded WUMI2024a_1993-0000000000-0000082944.tif
Downloaded WUMI2024a_1993-0000000000-0000092160.tif
Downloaded WUMI2024a_1993-0000009216-0000009216.tif
Downloaded WUMI2024a_1993-0000009216-0000036864.tif
Downloaded WUMI2024a_1993-0000009216-0000027648.tif
Downloaded WUMI2024a_1993-0000009216-0000018432.tif
Downloaded WUMI2024a_1993-0000009216-0000046080.tif
Downloaded WUMI2024a_19

'C:\\Users\\tymc5571\\dev\\cbi-module\\data\\derived\\WUMI2024a\\TEST_WUMI2024a_1993.tif'

In [35]:
from typing import Optional, List, Dict, Any
import os
from osgeo import gdal

gdal.UseExceptions()



def validate_raster_output(
    path: str,
    *,
    expect_cog: Optional[bool] = None,
    expect_compression: Optional[str] = None,
    expect_predictor: Optional[int] = None,
    expect_blocksize: Optional[int] = None,
    expect_overviews: Optional[bool] = None,
    expect_band_names: Optional[List[str]] = None,
    expect_band_count: Optional[int] = None,
    expect_nodata: Optional[float] = None,
    expect_crs_substring: Optional[str] = None,
    require_dataset_metadata_keys: Optional[List[str]] = None,
    fail_on_error: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Validate a raster output intended for GTiff or COG publishing.

    Returns a nested dictionary with:
      - summary
      - checks
      - dataset
      - bands
      - warnings
      - errors

    Metadata is returned in:
      - result["metadata"]["dataset"]
      - result["metadata"]["bands"]
    """

    def _safe_get(obj, domain: str = "") -> Dict[str, str]:
        md = obj.GetMetadata(domain)
        return md if md is not None else {}

    def _safe_domains(obj) -> List[str]:
        domains = obj.GetMetadataDomainList()
        return list(domains) if domains is not None else []

    def _all_metadata_by_domain(obj) -> Dict[str, Dict[str, str]]:
        out: Dict[str, Dict[str, str]] = {"default": _safe_get(obj)}
        for domain in _safe_domains(obj):
            key = "default" if domain == "" else domain
            out[key] = _safe_get(obj, domain)
        return out

    def _normalize_text(x):
        return None if x is None else str(x).strip()

    result: Dict[str, Any] = {
        "path": os.path.abspath(path),
        "summary": {},
        "checks": {},
        "dataset": {},
        "bands": [],
        "metadata": {
            "dataset": {},
            "bands": [],
        },
        "warnings": [],
        "errors": [],
    }

    def _record_check(name: str, ok: bool, detail: Any = None):
        result["checks"][name] = {"ok": bool(ok), "detail": detail}
        if not ok:
            result["errors"].append(f"{name}: {detail}")

    if not os.path.exists(path):
        msg = f"File does not exist: {path}"
        result["errors"].append(msg)
        if fail_on_error:
            raise FileNotFoundError(msg)
        return result

    ds = gdal.Open(path, gdal.GA_ReadOnly)
    if ds is None:
        msg = f"GDAL could not open file: {path}"
        result["errors"].append(msg)
        if fail_on_error:
            raise RuntimeError(msg)
        return result

    try:
        driver_name = ds.GetDriver().ShortName
        xsize = ds.RasterXSize
        ysize = ds.RasterYSize
        band_count = ds.RasterCount
        projection_wkt = ds.GetProjectionRef()
        geotransform = ds.GetGeoTransform(can_return_null=True)

        dataset_md = _all_metadata_by_domain(ds)
        image_structure = dataset_md.get("IMAGE_STRUCTURE", {})
        tiff_metadata = dataset_md.get("TIFF", {})

        result["dataset"]["driver"] = driver_name
        result["dataset"]["xsize"] = xsize
        result["dataset"]["ysize"] = ysize
        result["dataset"]["band_count"] = band_count
        result["dataset"]["projection_wkt"] = projection_wkt
        result["dataset"]["geotransform"] = geotransform
        result["dataset"]["metadata_domains"] = _safe_domains(ds)
        result["dataset"]["block_shapes"] = []
        result["dataset"]["overview_counts"] = []
        result["dataset"]["overview_sizes"] = []
        result["dataset"]["compression"] = image_structure.get("COMPRESSION")
        result["dataset"]["predictor"] = image_structure.get("PREDICTOR")
        result["dataset"]["interleave"] = image_structure.get("INTERLEAVE")
        result["dataset"]["layout"] = image_structure.get("LAYOUT")
        result["dataset"]["tiff_metadata"] = tiff_metadata

        result["metadata"]["dataset"] = dataset_md

        _record_check("can_open", True, "GDAL opened successfully")
        _record_check("has_bands", band_count > 0, f"band_count={band_count}")
        _record_check("has_size", xsize > 0 and ysize > 0, f"{xsize} x {ysize}")

        if expect_band_count is not None:
            _record_check(
                "expected_band_count",
                band_count == expect_band_count,
                f"expected={expect_band_count}, actual={band_count}",
            )

        if expect_crs_substring is not None:
            crs_ok = projection_wkt is not None and expect_crs_substring in projection_wkt
            _record_check(
                "expected_crs",
                crs_ok,
                f"expected substring={expect_crs_substring}",
            )

        is_cog_layout = image_structure.get("LAYOUT") == "COG" if image_structure else False

        block_shapes = []
        tiled_by_blocks = True
        for i in range(1, band_count + 1):
            band = ds.GetRasterBand(i)
            bx, by = band.GetBlockSize()
            block_shapes.append((bx, by))
            if by == 1:
                tiled_by_blocks = False

        is_tiled = tiled_by_blocks
        result["dataset"]["block_shapes"] = block_shapes

        unique_block_shapes = sorted(set(block_shapes))
        primary_block_shape = unique_block_shapes[0] if len(unique_block_shapes) == 1 else None

        _record_check("is_tiled", is_tiled, f"block_shapes={block_shapes}")

        if expect_blocksize is not None:
            block_ok = all(
                bx == expect_blocksize and by == expect_blocksize
                for bx, by in block_shapes
            )
            _record_check(
                "expected_blocksize",
                block_ok,
                f"expected={expect_blocksize}, actual={block_shapes}",
            )

        if expect_cog is not None:
            _record_check(
                "expected_cog",
                is_cog_layout == expect_cog,
                f"LAYOUT={image_structure.get('LAYOUT')}",
            )

        compression = image_structure.get("COMPRESSION")
        predictor = image_structure.get("PREDICTOR")

        if expect_compression is not None:
            comp_ok = _normalize_text(compression) == _normalize_text(expect_compression)
            _record_check(
                "expected_compression",
                comp_ok,
                f"expected={expect_compression}, actual={compression}",
            )

        if expect_predictor is not None:
            pred_ok = _normalize_text(predictor) == str(expect_predictor)
            _record_check(
                "expected_predictor",
                pred_ok,
                f"expected={expect_predictor}, actual={predictor}",
            )

        overview_counts = []
        overview_sizes = []
        overview_factors = []

        actual_band_names = []

        for i in range(1, band_count + 1):
            band = ds.GetRasterBand(i)

            desc = band.GetDescription()
            unit = band.GetUnitType()
            nodata = band.GetNoDataValue()
            dtype_name = gdal.GetDataTypeName(band.DataType)
            bx, by = band.GetBlockSize()

            n_ovr = band.GetOverviewCount()
            overview_counts.append(n_ovr)

            band_ovr_sizes = []
            band_ovr_factors = []
            for j in range(n_ovr):
                ovr = band.GetOverview(j)
                ox, oy = ovr.XSize, ovr.YSize
                band_ovr_sizes.append((ox, oy))

                fx = xsize / ox if ox else None
                fy = ysize / oy if oy else None
                band_ovr_factors.append((fx, fy))

            overview_sizes.append(band_ovr_sizes)
            overview_factors.append(band_ovr_factors)

            band_md = _all_metadata_by_domain(band)

            band_info = {
                "band": i,
                "description": desc,
                "unit": unit,
                "nodata": nodata,
                "dtype": dtype_name,
                "blocksize": (bx, by),
                "overview_count": n_ovr,
                "overview_sizes": band_ovr_sizes,
                "overview_factors": band_ovr_factors,
                "metadata_domains": _safe_domains(band),
            }
            result["bands"].append(band_info)
            result["metadata"]["bands"].append({
                "band": i,
                "metadata_by_domain": band_md,
            })

            actual_band_names.append(desc)

        has_overviews = all(v > 0 for v in overview_counts) if overview_counts else False
        result["dataset"]["overview_counts"] = overview_counts
        result["dataset"]["overview_sizes"] = overview_sizes
        result["dataset"]["overview_factors"] = overview_factors
        result["dataset"]["has_overviews_all_bands"] = has_overviews

        # Summary-friendly overview representation from band 1
        summary_overview_sizes = overview_sizes[0] if overview_sizes else []
        summary_overview_factors = overview_factors[0] if overview_factors else []

        if expect_overviews is not None:
            _record_check(
                "expected_overviews",
                has_overviews == expect_overviews,
                f"overview_counts={overview_counts}",
            )

        if require_dataset_metadata_keys:
            md_default = dataset_md.get("default", {})
            missing = [k for k in require_dataset_metadata_keys if k not in md_default]
            _record_check(
                "required_dataset_metadata_keys",
                len(missing) == 0,
                f"missing={missing}",
            )

        if expect_band_names is not None:
            band_names_ok = actual_band_names == expect_band_names
            _record_check(
                "expected_band_names",
                band_names_ok,
                f"expected={expect_band_names}, actual={actual_band_names}",
            )

        unnamed_bands = [
            b["band"] for b in result["bands"]
            if _normalize_text(b["description"]) in (None, "")
        ]
        _record_check(
            "all_bands_named",
            len(unnamed_bands) == 0,
            f"unnamed_bands={unnamed_bands}",
        )

        if expect_nodata is not None:
            nodata_ok = all(b["nodata"] == expect_nodata for b in result["bands"])
            _record_check(
                "expected_nodata",
                nodata_ok,
                f"expected={expect_nodata}, actual={[b['nodata'] for b in result['bands']]}",
            )

        result["summary"] = {
            "driver": driver_name,
            "size": (xsize, ysize),
            "band_count": band_count,
            "is_tiled": is_tiled,
            "tile_size": primary_block_shape,
            "tile_sizes_all_bands": unique_block_shapes,
            "has_overviews": has_overviews,
            "overview_count_band1": overview_counts[0] if overview_counts else 0,
            "overview_sizes_band1": summary_overview_sizes,
            "overview_factors_band1": summary_overview_factors,
            "layout": image_structure.get("LAYOUT"),
            "compression": compression,
            "predictor": predictor,
        }

        if expect_cog and not is_cog_layout:
            result["warnings"].append(
                "COG layout metadata not explicitly reported; file may still be usable, "
                "but this did not confirm byte-level COG layout."
            )

        if compression is None:
            result["warnings"].append(
                "Compression not exposed in IMAGE_STRUCTURE metadata on this GDAL build."
            )

        if predictor is None and expect_predictor is not None:
            result["warnings"].append(
                "Predictor was not exposed in IMAGE_STRUCTURE metadata, so that check may be inconclusive."
            )

        if fail_on_error and result["errors"]:
            raise RuntimeError(
                "Raster validation failed:\n- " + "\n- ".join(result["errors"])
            )

        if verbose:
            print(f"Validation summary for: {result['path']}")
            print(f"  Driver:             {result['summary']['driver']}")
            print(f"  Size:               {result['summary']['size'][0]} x {result['summary']['size'][1]}")
            print(f"  Bands:              {result['summary']['band_count']}")
            print(f"  Tiled:              {result['summary']['is_tiled']}")
            print(f"  Tile size:          {result['summary']['tile_size']}")
            print(f"  Tile sizes (all):   {result['summary']['tile_sizes_all_bands']}")
            print(f"  Overviews:          {result['summary']['has_overviews']}")
            print(f"  Overview count b1:  {result['summary']['overview_count_band1']}")
            print(f"  Overview sizes b1:  {result['summary']['overview_sizes_band1']}")
            print(f"  Overview factors b1:{result['summary']['overview_factors_band1']}")
            print(f"  Layout:             {result['summary']['layout']}")
            print(f"  Compression:        {result['summary']['compression']}")
            print(f"  Predictor:          {result['summary']['predictor']}")
            print(f"  Metadata domains:   {result['dataset']['metadata_domains']}")
            if result["warnings"]:
                print("  Warnings:")
                for w in result["warnings"]:
                    print(f"    - {w}")
            if result["errors"]:
                print("  Failed checks:")
                for e in result["errors"]:
                    print(f"    - {e}")
            else:
                print("  All requested checks passed.")

        return result

    finally:
        ds = None


report = validate_raster_output(
    os.path.join(DOWNLOAD_LOCATION, "TEST_WUMI2024a_1993.tif"),
    expect_cog=True,
    expect_compression="ZSTD",
    expect_predictor=3,
    expect_blocksize=512,
    expect_overviews=True,
    verbose=True,
)

report["metadata"]["dataset"]
report["metadata"]["bands"]
report["summary"]

Validation summary for: C:\Users\tymc5571\dev\cbi-module\data\derived\WUMI2024a\TEST_WUMI2024a_1993.tif
  Driver:             GTiff
  Size:               105324 x 104235
  Bands:              13
  Tiled:              True
  Tile size:          (512, 512)
  Tile sizes (all):   [(512, 512)]
  Overviews:          True
  Overview count b1:  8
  Overview sizes b1:  [(52662, 52117), (26331, 26058), (13165, 13029), (6582, 6514), (3291, 3257), (1645, 1628), (822, 814), (411, 407)]
  Overview factors b1:[(2.0, 2.0000191875971374), (4.0, 4.000115127791849), (8.000303835928598, 8.000230255583698), (16.001823154056517, 16.001688670555726), (32.003646308113034, 32.00337734111145), (64.02674772036474, 64.02641277641278), (128.13138686131387, 128.05282555282557), (256.26277372262774, 256.10565110565113)]
  Layout:             COG
  Compression:        ZSTD
  Predictor:          3
  Metadata domains:   ['IMAGE_STRUCTURE', '', 'DERIVED_SUBDATASETS']
  All requested checks passed.


{'driver': 'GTiff',
 'size': (105324, 104235),
 'band_count': 13,
 'is_tiled': True,
 'tile_size': (512, 512),
 'tile_sizes_all_bands': [(512, 512)],
 'has_overviews': True,
 'overview_count_band1': 8,
 'overview_sizes_band1': [(52662, 52117),
  (26331, 26058),
  (13165, 13029),
  (6582, 6514),
  (3291, 3257),
  (1645, 1628),
  (822, 814),
  (411, 407)],
 'overview_factors_band1': [(2.0, 2.0000191875971374),
  (4.0, 4.000115127791849),
  (8.000303835928598, 8.000230255583698),
  (16.001823154056517, 16.001688670555726),
  (32.003646308113034, 32.00337734111145),
  (64.02674772036474, 64.02641277641278),
  (128.13138686131387, 128.05282555282557),
  (256.26277372262774, 256.10565110565113)],
 'layout': 'COG',
 'compression': 'ZSTD',
 'predictor': '3'}